
# Exploración de PokéAPI — Minería de Datos

**Proyecto:** Pokémon Team Recommender  
**Fase:** 1 — Minería de Datos  
**Fuente:** https://pokeapi.co/api/v2/

## Objetivo

Explorar los recursos de PokéAPI y verificar la disponibilidad, estructura y consistencia preliminar de los datos necesarios para caracterizar Pokémon y diseñar un recomendador de equipos de tres integrantes.

### Actividades

1. Comprobar la conexión con PokéAPI.
2. Explorar la paginación.
3. Inspeccionar estadísticas, tipos y habilidades.
4. Consultar información de especies.
5. Identificar variedades.
6. Analizar relaciones de efectividad de tipos.
7. Realizar controles iniciales de calidad sobre una muestra.

**Alcance:** exploración controlada, sin extracción masiva.


In [1]:
import sys
import pandas as pd
import numpy as np
import requests
import sklearn

print("Kernel funcionando correctamente")
print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Kernel funcionando correctamente
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Ejecutable: c:\Users\REYES\OneDrive\Desktop\pokemon team recommended\.venv\Scripts\python.exe
Pandas: 3.0.5
NumPy: 2.5.3
Scikit-learn: 1.9.0


In [2]:

import requests
import pandas as pd

from datetime import datetime, timezone
from pprint import pprint

BASE_URL = "https://pokeapi.co/api/v2"

session = requests.Session()
session.headers.update({
    "User-Agent": "pokemon-team-recommender/0.1",
    "Accept": "application/json"
})


def get_resource(endpoint):
    """Consulta un recurso de PokéAPI y devuelve su JSON."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"

    response = session.get(url, timeout=20)
    response.raise_for_status()

    return response.json()


print("Configuración inicial completada")
print("Fecha UTC:", datetime.now(timezone.utc).isoformat())
print("URL base:", BASE_URL)


Configuración inicial completada
Fecha UTC: 2026-11-03T09:14:16.780200+00:00
URL base: https://pokeapi.co/api/v2



## 1. Conectividad y paginación

Se verificará el acceso a PokéAPI mediante una consulta limitada al recurso `pokemon`.

El endpoint de listado proporciona información de paginación mediante:

- `count`: cantidad total de resultados informada por la API.
- `next`: dirección de la siguiente página.
- `previous`: dirección de la página anterior.
- `results`: elementos recuperados.

La cantidad total reportada no se interpretará como el número de especies únicas.


In [3]:

pokemon_list = get_resource("pokemon?limit=5&offset=0")

print("Total reportado:", pokemon_list["count"])
print("Siguiente página:", pokemon_list["next"])

df_list = pd.DataFrame(pokemon_list["results"])

display(df_list)


Total reportado: 1351
Siguiente página: https://pokeapi.co/api/v2/pokemon?offset=5&limit=5


,name,url
0,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/
1,ivysaur,https://pokeapi.co/api/v2/pokemon/2/
2,venusaur,https://pokeapi.co/api/v2/pokemon/3/
3,charmander,https://pokeapi.co/api/v2/pokemon/4/
4,charmeleon,https://pokeapi.co/api/v2/pokemon/5/



## 2. Exploración del recurso Pokémon

Se inspeccionará una entrada individual para identificar:

- Identificadores.
- Estadísticas base.
- Tipos.
- Habilidades.
- Cantidad de movimientos asociados.
- Relación con la especie.

Se utilizará Bulbasaur (`pokemon_id = 1`) como ejemplo exploratorio.

La presencia de datos en este ejemplo no garantiza su completitud para toda la población.


In [4]:

pokemon = get_resource("pokemon/1")

print("ID:", pokemon["id"])
print("Nombre:", pokemon["name"])
print("Especie:", pokemon["species"]["name"])
print("Variedad predeterminada:", pokemon["is_default"])
print("Altura (dm):", pokemon["height"])
print("Peso (hg):", pokemon["weight"])

stats = {
    item["stat"]["name"]: item["base_stat"]
    for item in pokemon["stats"]
}

types = [
    item["type"]["name"]
    for item in sorted(
        pokemon["types"],
        key=lambda item: item["slot"]
    )
]

abilities = [
    {
        "ability": item["ability"]["name"],
        "is_hidden": item["is_hidden"]
    }
    for item in pokemon["abilities"]
]

print("\nEstadísticas:")
pprint(stats)

print("\nTipos:")
pprint(types)

print("\nHabilidades:")
pprint(abilities)

print("\nCantidad de movimientos relacionados:", len(pokemon["moves"]))


ID: 1
Nombre: bulbasaur
Especie: bulbasaur
Variedad predeterminada: True
Altura (dm): 7
Peso (hg): 69

Estadísticas:
{'attack': 49,
 'defense': 49,
 'hp': 45,
 'special-attack': 65,
 'special-defense': 65,
 'speed': 45}

Tipos:
['grass', 'poison']

Habilidades:
[{'ability': 'overgrow', 'is_hidden': False},
 {'ability': 'chlorophyll', 'is_hidden': True}]

Cantidad de movimientos relacionados: 86



## 3. Exploración de Pokémon Species

El recurso `pokemon-species` contiene información diferente del recurso `pokemon`.

Se analizarán:

- Generación.
- Clasificación legendaria y mítica.
- Tasa de captura.
- Hábitat.
- Cadena evolutiva.
- Variedades relacionadas.

La relación entre especies y entradas Pokémon será importante para evitar duplicados durante la ingeniería de datos.


In [5]:

species = get_resource("pokemon-species/1")

species_info = {
    "species_id": species["id"],
    "species_name": species["name"],
    "generation": species["generation"]["name"],
    "is_legendary": species["is_legendary"],
    "is_mythical": species["is_mythical"],
    "capture_rate": species["capture_rate"],
    "habitat": (
        species["habitat"]["name"]
        if species["habitat"] is not None
        else None
    ),
    "evolution_chain_url": (
        species["evolution_chain"]["url"]
        if species["evolution_chain"] is not None
        else None
    ),
    "varieties_count": len(species["varieties"])
}

display(pd.DataFrame([species_info]))


,species_id,species_name,generation,is_legendary,is_mythical,capture_rate,habitat,evolution_chain_url,varieties_count
0,1,bulbasaur,generation-i,False,False,45,grassland,https://pokeapi.co/api/v2/evolution-chain/1/,1



## 4. Diferencias entre especies y variedades

Una especie puede estar relacionada con varias entradas Pokémon.

Esto es relevante porque algunas variedades pueden presentar estadísticas, tipos o habilidades diferentes.

Para comprobarlo se explorará la especie Charizard.

La unidad analítica preliminar continuará siendo `pokemon_id`, mientras que `species_id` permitirá identificar la especie de origen.


In [6]:

fire_type = get_resource("type/fire")

damage_relations = fire_type["damage_relations"]

relation_names = [
    "double_damage_to",
    "half_damage_to",
    "no_damage_to",
    "double_damage_from",
    "half_damage_from",
    "no_damage_from"
]

for relation in relation_names:
    values = [
        item["name"]
        for item in damage_relations[relation]
    ]

    print(f"{relation}: {values}")


double_damage_to: ['bug', 'steel', 'grass', 'ice']
half_damage_to: ['rock', 'fire', 'water', 'dragon']
no_damage_to: []
double_damage_from: ['ground', 'rock', 'water']
half_damage_from: ['bug', 'steel', 'fire', 'grass', 'ice', 'fairy']
no_damage_from: []



## 6. Evaluación preliminar de calidad

Se realizará una revisión exploratoria sobre una muestra pequeña de entradas Pokémon.

### Muestra

Identificadores:

`1, 4, 7, 25, 133, 150`

### Controles

1. Identificador disponible.
2. Nombre disponible.
3. Referencia a especie.
4. Presencia de las seis estadísticas base.
5. Presencia de tipos.
6. Identificadores duplicados.
7. Valores nulos.

**Limitación:** esta muestra es intencional y no representativa de toda la población. Los resultados no podrán generalizarse al dataset completo.


In [7]:

sample_ids = [1, 4, 7, 25, 133, 150]

expected_stats = {
    "hp",
    "attack",
    "defense",
    "special-attack",
    "special-defense",
    "speed"
}

records = []

for pokemon_id in sample_ids:
    item = get_resource(f"pokemon/{pokemon_id}")

    observed_stats = {
        stat["stat"]["name"]
        for stat in item["stats"]
    }

    records.append({
        "pokemon_id": item["id"],
        "pokemon_name": item["name"],
        "species_name": item["species"]["name"],
        "stats_complete": expected_stats.issubset(observed_stats),
        "stats_count": len(observed_stats),
        "types_count": len(item["types"]),
        "abilities_count": len(item["abilities"]),
        "moves_count": len(item["moves"]),
        "is_default": item["is_default"]
    })

df_quality = pd.DataFrame(records)

print("Registros analizados:", len(df_quality))
print("IDs duplicados:", df_quality["pokemon_id"].duplicated().sum())
print("Registros con seis estadísticas:", df_quality["stats_complete"].sum())
print("\nValores nulos por columna:")
print(df_quality.isna().sum())

display(df_quality)


Registros analizados: 6
IDs duplicados: 0
Registros con seis estadísticas: 6

Valores nulos por columna:
pokemon_id         0
pokemon_name       0
species_name       0
stats_complete     0
stats_count        0
types_count        0
abilities_count    0
moves_count        0
is_default         0
dtype: int64


,pokemon_id,pokemon_name,species_name,stats_complete,stats_count,types_count,abilities_count,moves_count,is_default
0,1,bulbasaur,bulbasaur,True,6,2,2,86,True
1,4,charmander,charmander,True,6,1,2,105,True
2,7,squirtle,squirtle,True,6,1,2,105,True
3,25,pikachu,pikachu,True,6,1,2,109,True
4,133,eevee,eevee,True,6,1,3,74,True
5,150,mewtwo,mewtwo,True,6,1,2,167,True



## 7. Diagnóstico preliminar

Los resultados de este notebook permitirán comprobar empíricamente la estructura de los recursos seleccionados.

### Aspectos que deben revisarse

- ¿La API respondió correctamente?
- ¿Las estadísticas esperadas están presentes?
- ¿La información de especies puede relacionarse con Pokémon?
- ¿Existen variedades asociadas a una misma especie?
- ¿Las relaciones de tipos pueden utilizarse para construir una matriz de efectividad?
- ¿Se observaron valores faltantes o duplicados en la muestra?

### Limitaciones

La exploración se realizó sobre un conjunto reducido de recursos.

No representa una auditoría de calidad de toda la población.

Tampoco demuestra la efectividad del futuro recomendador.

### Siguiente etapa

Consolidar los resultados en `mineria/docs/05_diagnostico_final.md` y definir los requisitos de extracción de la fase de ingeniería de datos.
